In [22]:
import pandas as pd

In [23]:
# Path to GLSEA SST data
ver_file = '/Users/ljob/Desktop/CNBS_forecast_ver_combined.csv'
bias_file = '/Users/ljob/Desktop/CNBS_forecast_ver_swe.csv'

ver_data = pd.read_csv(ver_file,sep='\t')
bias_data = pd.read_csv(bias_file,sep='\t')


In [24]:
import pandas as pd
import numpy as np

META_COLS = ["cfs_run", "model", "forecast_month"]

# ----------------------------
# METRIC
# ----------------------------
def compute_skill_grouped(df, metric="rmse"):
    if metric == "rmse":
        return np.sqrt(np.mean((df["obs"] - df["forecast"])**2))
    elif metric == "mae":
        return np.mean(np.abs(df["obs"] - df["forecast"]))
    elif metric == "corr":
        return df["obs"].corr(df["forecast"])
    else:
        raise ValueError("Unsupported metric")


# ----------------------------
# RESHAPE
# ----------------------------
def reshape_long(df):
    df = df.copy()
    df["forecast_month"] = pd.to_datetime(df["forecast_month"])

    value_cols = [c for c in df.columns if c not in META_COLS]

    long_df = df.melt(
        id_vars=META_COLS,
        value_vars=value_cols,
        var_name="variable",
        value_name="value"
    )

    long_df["type"] = np.where(
        long_df["variable"].str.endswith("_obs"),
        "obs",
        "forecast"
    )

    long_df["variable_clean"] = long_df["variable"].str.replace("_obs", "", regex=False)

    split = long_df["variable_clean"].str.split("_", n=1, expand=True)
    long_df["lake"] = split[0]
    long_df["component"] = split[1]

    # 👇 include model in pivot index
    wide = long_df.pivot_table(
        index=["forecast_month", "cfs_run", "model", "lake", "component"],
        columns="type",
        values="value"
    ).reset_index()

    return wide


# ----------------------------
# SKILL
# ----------------------------
def compute_skill(df_long, metric="rmse"):
    skill = (
        df_long
        .dropna(subset=["forecast", "obs"])
        .groupby(["forecast_month", "model", "lake", "component"])
        .apply(lambda g: compute_skill_grouped(g, metric))
        .reset_index(name="skill")
    )

    return skill


# ----------------------------
# COMPARE
# ----------------------------
def compare_datasets(df1, df2, metric="rmse"):
    long1 = reshape_long(df1)
    long2 = reshape_long(df2)

    skill1 = compute_skill(long1, metric).rename(columns={"skill": "skill_A"})
    skill2 = compute_skill(long2, metric).rename(columns={"skill": "skill_B"})

    merged = pd.merge(
        skill1,
        skill2,
        on=["forecast_month", "model", "lake", "component"],
        how="inner"
    )

    if metric in ["rmse", "mae"]:
        merged["improvement_pct"] = (
            (merged["skill_A"] - merged["skill_B"]) / merged["skill_A"] * 100
        )
    else:
        merged["improvement_pct"] = (
            (merged["skill_B"] - merged["skill_A"]) / np.abs(merged["skill_A"]) * 100
        )

    merged["winner"] = np.where(
        merged["improvement_pct"] > 0,
        "Dataset B",
        "Dataset A"
    )

    return merged


# ----------------------------
# SUMMARYS
# ----------------------------

# overall by model/lake/component
def summarize_results(comparison_df):
    return (
        comparison_df
        .groupby(["model", "lake", "component"])
        .agg(
            skill_A=("skill_A", "mean"),
            skill_B=("skill_B", "mean"),
            improvement_pct=("improvement_pct", "mean")
        )
        .reset_index()
        .sort_values("improvement_pct", ascending=False)
    )


# ----------------------------
# USAGE
# ----------------------------
comparison = compare_datasets(ver_data, bias_data, metric="rmse")

summary = summarize_results(comparison)


/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_19125/3447268616.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: compute_skill_grouped(g, metric))
/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_19125/3447268616.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: compute_skill_grouped(g, metric))


In [25]:

print("\n=== By Model/Lake/Component ===")
print(
    summary[
        ["model", "lake", "component", "skill_A", "skill_B", "improvement_pct"]
    ]
)


# which model benefits most
def summarize_by_model(comparison_df):
    return (
        comparison_df
        .groupby("model")["improvement_pct"]
        .mean()
        .reset_index()
        .sort_values("improvement_pct", ascending=False)
    )


# ----------------------------
# USAGE
# ----------------------------
comparison = compare_datasets(ver_data, bias_data, metric="rmse")

summary = summarize_results(comparison)
model_summary = summarize_by_model(comparison)

print("\n=== By Model/Lake/Component ===")
print(summary)

print("\n=== Model Ranking ===")
print(model_summary)


=== By Model/Lake/Component ===
   model            lake      component    skill_A    skill_B  improvement_pct
2     GP            erie  precipitation  34.992087  22.568058        40.991783
6     GP  michigan-huron  precipitation  31.047579  19.723962        40.308333
14    GP        superior  precipitation  29.117269  19.128952        38.721208
7     GP  michigan-huron         runoff  19.382737  12.673165        37.645368
5     GP  michigan-huron            nbs  55.387127  36.571054        36.139294
..   ...             ...            ...        ...        ...              ...
51   XGB            erie         runoff  32.612523  36.874492       -25.160577
35    RF            erie         runoff  24.719233  25.154466       -29.696095
31    NN        superior         runoff  15.112898  17.285270       -31.943655
59   XGB         ontario         runoff  46.061442  52.536380       -36.990056
63   XGB        superior         runoff  13.948373  20.008545      -103.674700

[64 rows x 6 colum

/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_19125/3447268616.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: compute_skill_grouped(g, metric))
/var/folders/2w/ddc7n0594ydfswtzw_mmfs6c0000gp/T/ipykernel_19125/3447268616.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: compute_skill_grouped(g, metric))



=== By Model/Lake/Component ===
   model            lake      component    skill_A    skill_B  improvement_pct
2     GP            erie  precipitation  34.992087  22.568058        40.991783
6     GP  michigan-huron  precipitation  31.047579  19.723962        40.308333
14    GP        superior  precipitation  29.117269  19.128952        38.721208
7     GP  michigan-huron         runoff  19.382737  12.673165        37.645368
5     GP  michigan-huron            nbs  55.387127  36.571054        36.139294
..   ...             ...            ...        ...        ...              ...
51   XGB            erie         runoff  32.612523  36.874492       -25.160577
35    RF            erie         runoff  24.719233  25.154466       -29.696095
31    NN        superior         runoff  15.112898  17.285270       -31.943655
59   XGB         ontario         runoff  46.061442  52.536380       -36.990056
63   XGB        superior         runoff  13.948373  20.008545      -103.674700

[64 rows x 6 colum

In [26]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(summary)

   model            lake      component     skill_A     skill_B  \
2     GP            erie  precipitation   34.992087   22.568058   
6     GP  michigan-huron  precipitation   31.047579   19.723962   
14    GP        superior  precipitation   29.117269   19.128952   
7     GP  michigan-huron         runoff   19.382737   12.673165   
5     GP  michigan-huron            nbs   55.387127   36.571054   
10    GP         ontario  precipitation   32.290474   22.266673   
12    GP        superior    evaporation   18.759330   11.497205   
13    GP        superior            nbs   54.268481   36.145637   
0     GP            erie    evaporation   21.315070   14.047819   
4     GP  michigan-huron    evaporation   17.856733   12.334693   
1     GP            erie            nbs  123.380670   93.493271   
3     GP            erie         runoff   32.927395   25.297058   
48   XGB            erie    evaporation   27.089628   17.970439   
15    GP        superior         runoff   14.215772   10.34201